<a href="https://colab.research.google.com/github/JamesMartinOU/PublicRedditSentimentAnalysis/blob/main/WriteFactTablesToSnowflake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Python libraries
!pip install mysql-connector-python pandas sqlalchemy snowflake-connector-python snowflake.sqlalchemy

In [12]:
# Import Python libraries
import mysql.connector
import pandas as pd
from sqlalchemy import create_engine
import snowflake.connector
from snowflake.sqlalchemy import URL
import datetime

In [3]:
# RDS MySQL connection details


In [4]:
# Snowflake Credentials

In [5]:
# Define the query
query = """
SELECT tmp.company_name, tmp.keyword, tmp.keyword_label, tmp.year, tmp.week, tmp.sentiment, SUM(tmp.count) as count
FROM (
SELECT
  "comment" as type,
  rcsgb.company_name,
  rcsgb.keyword,
  CASE
    WHEN rcsgb.keyword = rcsgb.company_name THEN "company_name"
    WHEN length(rcsgb.keyword) <= 5 THEN "stock_symbol"
    ELSE "ceo_name"
  END as keyword_label,
  year(rcsgb.created_utc) as year,
  week(rcsgb.created_utc, 1) as week,
  rcsgb.avg_sentiment as sentiment,
  count(rcsgb.comment_id) as count
FROM (
  SELECT
    arc.post_id,
    arc.comment_id,
    arc.created_utc,
    rp.company_name,
    rp.keyword,
    CASE
      WHEN avg(arc.sentiment_score) >= .33 THEN "Positive"
      WHEN avg(arc.sentiment_score) <= -.33 THEN "Negative"
      ELSE "Neutral"
    END as avg_sentiment
  FROM (
    SELECT
      rcs.post_id,
      rcs.comment_id,
      rc.created_utc,
      CASE
        WHEN rcs.sentiment = "Neutral" THEN 0
        WHEN rcs.sentiment = "Positive" THEN 1
        ELSE -1
      END as sentiment_score
    FROM reddit_comments_sentiment as rcs
    INNER JOIN reddit_comments as rc ON rc.comment_id = rcs.comment_id
  ) as arc
  INNER JOIN
    reddit_posts as rp ON rp.id = arc.post_id
  GROUP BY
    arc.post_id,
    arc.comment_id,
    arc.created_utc,
    rp.company_name,
    rp.keyword
) as rcsgb
GROUP BY
  rcsgb.company_name,
  rcsgb.keyword,
  keyword_label,
  year(rcsgb.created_utc),
  week,
  rcsgb.avg_sentiment

UNION ALL

SELECT
  "post" as type,
  rp.Company_Name,
  rp.Keyword,
  CASE
    WHEN rp.keyword = rp.company_name THEN "company_name"
    WHEN length(rp.keyword) <= 5 THEN "stock_symbol"
    ELSE "ceo_name"
  END as keyword_label,
  year(created_utc) as year,
  week(created_utc, 1) as week,
  rps.Sentiment,
  count(rp.id) as count
FROM reddit_posts as rp
INNER JOIN
  reddit_posts_sentiment as rps ON rps.id = rp.id
GROUP BY
  rp.Company_Name,
  rp.Keyword,
  keyword_label,
  year(created_utc),
  week,
  rps.Sentiment
  ) as tmp
GROUP BY
  tmp.company_name,
  tmp.keyword,
  tmp.keyword_label,
  tmp.year,
  tmp.week,
  tmp.sentiment
"""


In [10]:
query_rolling_avg_select = """
WITH RECURSIVE calendar AS (
  -- start with the earliest year/week
  SELECT
    MIN(year) AS year,
    MIN(week) AS week
  FROM reddit_sentiment_fact_table_weekly

  UNION ALL

  -- increment week, roll over year when needed
  SELECT
    CASE WHEN week = 52 THEN year + 1 ELSE year END,
    CASE WHEN week = 52 THEN 1 ELSE week + 1 END
  FROM calendar
  WHERE (year < (SELECT MAX(year) FROM reddit_sentiment_fact_table_weekly))
     OR (year = (SELECT MAX(year) FROM reddit_sentiment_fact_table_weekly)
         AND week < (SELECT MAX(week) FROM reddit_sentiment_fact_table_weekly))
),

company_weeks AS (
  SELECT DISTINCT company_name FROM reddit_sentiment_fact_table_weekly
),
all_weeks AS (
  SELECT
    cw.company_name,
    c.year,
    c.week
  FROM calendar c
  CROSS JOIN company_weeks cw
),

weekly_sentiment AS (
  SELECT
    company_name,
    year,
    week,
    SUM(
      CASE
        WHEN sentiment = 'Positive' THEN 1
        WHEN sentiment = 'Negative' THEN -1
        ELSE 0
      END * count
    ) AS total_score,
    SUM(count) AS total_count
  FROM reddit_sentiment_fact_table_weekly
  GROUP BY company_name, year, week
),

joined_timeline AS (
  SELECT
    aw.company_name,
    aw.year,
    aw.week,
    ws.total_score,
    ws.total_count
  FROM all_weeks aw
  LEFT JOIN weekly_sentiment ws
    ON aw.company_name = ws.company_name
   AND aw.year = ws.year
   AND aw.week = ws.week
)

SELECT
  company_name,
  year,
  week,
  total_score,
  total_count,
  ROUND(total_score / NULLIF(total_count, 0), 3) AS weekly_sentiment,

  -- 3-week rolling average
  ROUND(
    AVG(total_score * 1.0 / NULLIF(total_count, 0)) OVER (
      PARTITION BY company_name
      ORDER BY year, week
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),
    3
  ) AS rolling_avg_sentiment

FROM joined_timeline
ORDER BY company_name, year, week;

"""

In [ ]:
# MySQL connection setup
mysql_conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)


# Select from MySQL into df
df = pd.read_sql(query, con=mysql_conn)
df.columns = [col.lower() for col in df.columns]
mysql_conn.close()


# Snowflake Connection
sf_engine = create_engine(URL(
    account='ab08591.us-east-2.aws',
    user='JAMESMARTINOU',
    password='BedardDatsyuk1990',
    database='REDDITDB',
    schema='PUBLIC',
    warehouse='COMPUTE_WH',
    role='SYSADMIN'
))


# Insert df into Snowflake table
with sf_engine.connect() as conn:
    # Create or replace table with matching lowercase column names
    df.head(0).to_sql('reddit_sentiment_weekly', con=conn, index=False, if_exists='replace')
    # Append full data
    df.to_sql('reddit_sentiment_weekly', con=conn, index=False, if_exists='append', method='multi')


In [ ]:
# --- MySQL Connection ---
mysql_conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

cursor = mysql_conn.cursor()

# Execute rolling average query
cursor.execute(query_rolling_avg_select)
rows = cursor.fetchall()

# Create DataFrame from results
df_sentiment_fact = pd.DataFrame(rows, columns=[
    'company_name', 'year', 'week',
    'total_score', 'total_count',
    'weekly_sentiment', 'rolling_avg_sentiment'
])

# Fill missing rolling averages (forward-fill within each company)
df_sentiment_fact['rolling_avg_sentiment_filled'] = (
    df_sentiment_fact
    .groupby('company_name')['rolling_avg_sentiment']
    .transform(lambda x: x.fillna(method='ffill'))
)

# Optional: drop rows where fill wasn't possible (still NaN)
df_sentiment_fact.dropna(subset=['rolling_avg_sentiment_filled'], inplace=True)

# Normalize column names to lowercase
df_sentiment_fact.columns = [col.lower() for col in df_sentiment_fact.columns]

mysql_conn.close()

# Snowflake Connection
sf_engine = create_engine(URL(
    account='ab08591.us-east-2.aws',
    user='JAMESMARTINOU',
    password='BedardDatsyuk1990',
    database='REDDITDB',
    schema='PUBLIC',
    warehouse='COMPUTE_WH',
    role='SYSADMIN'
))

# Write to Snowflake
with sf_engine.connect() as conn:
    # Create table with no data first (ensures correct structure)
    df_sentiment_fact.head(0).to_sql('reddit_sentiment_rolling_average', con=conn, index=False, if_exists='replace')
    # Append full cleaned data
    df_sentiment_fact.to_sql('reddit_sentiment_rolling_average', con=conn, index=False, if_exists='append', method='multi')